# LeetCode 2327 — Number of People Aware of a Secret bằng Python

Link đề: https://leetcode.com/problems/number-of-people-aware-of-a-secret/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Queue, Simulation

Skeleton LeetCode:

```python
class Solution:
    def peopleAwareOfSecret(self, n: int, delay: int, forget: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển từ mô phỏng **từng người** (bùng nổ theo hàm mũ) sang đếm theo **nhóm cùng ngày biết bí mật**: `new[d]` = số người biết bí mật **lần đầu** vào ngày `d`.
2. Viết truy hồi: người biết ngày `j` chia sẻ vào các ngày `d ∈ [j + delay, j + forget − 1]` ⇒ `new[d] = Σ_{j = d − forget + 1}^{d − delay} new[j]`.
3. Tăng tốc từ O(n · forget) xuống **O(n)** bằng **cửa sổ trượt** / **tổng tiền tố** / **mảng hiệu**.
4. Đáp án = số người **còn nhớ** cuối ngày `n` = `Σ new[d]` với `d ∈ [n − forget + 1, n]`.
5. Cẩn thận các biên "đúng ngày quên thì không chia sẻ được nữa" và lấy dư khi trừ.

## 2. Đề bài & ví dụ

Ngày 1, một người biết một bí mật. Mỗi người bắt đầu chia sẻ bí mật cho **một người mới mỗi ngày**, kể từ `delay` ngày sau khi biết. Mỗi người **quên** bí mật `forget` ngày sau khi biết; không thể chia sẻ vào ngày quên hoặc sau đó. Trả về số người **biết** bí mật vào cuối ngày `n`, modulo `10⁹ + 7`.

**Ví dụ 1:** `n = 6, delay = 2, forget = 4` → `5`.

**Ví dụ 2:** `n = 4, delay = 1, forget = 3` → `6`.

**Ràng buộc:** `2 ≤ n ≤ 1000`; `1 ≤ delay < forget ≤ n`.

In [1]:
from collections import deque
import random, time

MOD = 10**9 + 7
examples = [(6, 2, 4, 5), (4, 1, 3, 6)]
for n, d, f, exp in examples:
    print(f"n={n} delay={d} forget={f} -> kỳ vọng {exp}")

n=6 delay=2 forget=4 -> kỳ vọng 5
n=4 delay=1 forget=3 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Dòng thời gian của một người biết bí mật ngày `j`

- Ngày `j … j + delay − 1`: biết, **chưa** chia sẻ.
- Ngày `j + delay … j + forget − 1`: biết và **mỗi ngày** chia sẻ cho một người mới.
- Ngày `j + forget` trở đi: đã quên (không tính là "biết", không chia sẻ).

In [2]:
def timeline(j, delay, forget, upto):
    out = []
    for d in range(j, upto + 1):
        if d < j + delay: out.append(f"{d}:biết")
        elif d < j + forget: out.append(f"{d}:chia sẻ")
        else: out.append(f"{d}:quên")
    return out
print(timeline(1, 2, 4, 6))

['1:biết', '2:biết', '3:chia sẻ', '4:chia sẻ', '5:quên', '6:quên']


### 3.2 Tổng tiền tố và cửa sổ trượt

`S[d] = Σ_{i ≤ d} new[i]` ⇒ tổng trên đoạn `[a, b]` là `S[b] − S[a − 1]` trong O(1). Hoặc giữ biến `share` = số người đang chia sẻ, cập nhật `+new[d − delay] − new[d − forget]` mỗi ngày.

### 3.2b Mảng hiệu (difference array)

Muốn cộng `c` cho mọi ngày trong `[a, b)`: `diff[a] += c`, `diff[b] −= c`; tổng tiền tố của `diff` cho giá trị từng ngày. Cách doocs dùng mảng hiệu để đếm "người còn nhớ": nhóm biết ngày `j` còn nhớ trong `[j, j + forget)`.

In [3]:
diff = [0] * 10
for a, b, c in [(1, 5, 1), (3, 7, 2)]:
    diff[a] += c; diff[b] -= c
vals = []; run = 0
for x in diff: run += x; vals.append(run)
print("giá trị từng ngày:", vals)

giá trị từng ngày: [0, 1, 1, 3, 3, 2, 2, 0, 0, 0]


### 3.3 Modulo khi trừ

`(x − y) mod p` trong Python luôn không âm; trong C++/Java cần `(x − y + p) % p`.

In [4]:
print((5 - 7) % MOD, "| Python % luôn cho kết quả không âm khi số chia dương")

1000000005 | Python % luôn cho kết quả không âm khi số chia dương


## 4. Ý tưởng nền tảng / trực giác

Những người biết bí mật **cùng ngày** hành xử giống hệt nhau ⇒ chỉ cần đếm `new[d]`.

Người biết ngày `j` chia sẻ đúng vào ngày `d` ⇔ `j + delay ≤ d ≤ j + forget − 1` ⇔ `d − forget + 1 ≤ j ≤ d − delay`. Mỗi người đang chia sẻ tạo ra một người mới ⇒

`new[d] = Σ_{j = max(1, d − forget + 1)}^{d − delay} new[j]`, `new[1] = 1`.

Người còn nhớ cuối ngày `n` là người biết vào ngày `d` với `d + forget > n` ⇔ `d ≥ n − forget + 1`.

Tính tay ví dụ 1 (`delay = 2`, `forget = 4`):

| ngày d | ai đang chia sẻ (ngày biết j ∈ [d−3, d−2]) | new[d] | còn nhớ cuối ngày (j ∈ [d−3, d]) |
|---|---|---|---|
| 1 | — | 1 (A) | 1 |
| 2 | j ∈ [−1, 0] | 0 | 1 |
| 3 | j = 1 | 1 (B) | 2 |
| 4 | j ∈ [1, 2] | 1 (C) | 3 |
| 5 | j ∈ [2, 3] | 1 (D) | new[2..5] = 3 |
| 6 | j ∈ [3, 4] | 2 (E, F) | new[3..6] = 5 |

⇒ `5` ✔.

In [5]:
n, delay, forget, _ = examples[0]
new = [0] * (n + 1); new[1] = 1
for d in range(2, n + 1):
    new[d] = sum(new[j] for j in range(max(1, d - forget + 1), d - delay + 1))
    know = sum(new[max(1, d - forget + 1): d + 1])
    print(f"ngày {d}: new = {new[d]}, còn nhớ = {know}")

ngày 2: new = 0, còn nhớ = 1
ngày 3: new = 1, còn nhớ = 2
ngày 4: new = 1, còn nhớ = 3
ngày 5: new = 1, còn nhớ = 3
ngày 6: new = 2, còn nhớ = 5


## 5. Cách 1 — Brute Force / baseline

### 5.1 Mô phỏng **từng người** (danh sách ngày biết bí mật) — số người tăng theo hàm mũ

In [6]:
def brute(n, delay, forget):
    people = [1]                                   # ngày biết bí mật của từng người
    for d in range(2, n + 1):
        sharers = sum(1 for j in people if j + delay <= d < j + forget)
        people.extend([d] * sharers)
        people = [j for j in people if j + forget > d]   # bỏ người đã quên (giữ danh sách nhỏ hơn)
    return len(people) % MOD

for n, d, f, exp in examples:
    print(brute(n, d, f), "OK" if brute(n, d, f) == exp else "SAI")

5 OK
6 OK


Đúng theo định nghĩa nhưng danh sách có thể có tới hàng tỉ phần tử (ví dụ `delay = 1, forget = n`: số người gấp đôi mỗi ngày) — chỉ chạy với `n` nhỏ.

### 5.2 Baseline 2 — DP theo nhóm, cộng thẳng cửa sổ — O(n · forget)

In [7]:
def dp_quadratic(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1
    for d in range(2, n + 1):
        new[d] = sum(new[j] for j in range(max(1, d - forget + 1), d - delay + 1)) % MOD
    return sum(new[max(1, n - forget + 1): n + 1]) % MOD

for n, d, f, exp in examples:
    print(dp_quadratic(n, d, f), "OK" if dp_quadratic(n, d, f) == exp else "SAI")

5 OK
6 OK


Với `n = 1000` đã đủ nhanh (`≤ 10⁶` phép cộng), nhưng **nút thắt** là cửa sổ `[d − forget + 1, d − delay]` trượt sang phải đúng 1 mỗi ngày — tổng của nó cập nhật được trong O(1).

## 6. Ý tưởng thuật toán / bước đột phá

**Cửa sổ trượt:** `share` = số người đang chia sẻ vào ngày `d`.

```
new[1] = 1; share = 0
for d in 2..n:
    share += new[d − delay]          # nhóm vừa đến tuổi chia sẻ (nếu d − delay ≥ 1)
    share −= new[d − forget]         # nhóm vừa quên (nếu d − forget ≥ 1)
    new[d] = share
answer = Σ new[d] for d in [n − forget + 1, n]
```

**Độ phức tạp:** O(n) thời gian, O(n) bộ nhớ (có thể O(forget) bằng hàng đợi vòng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng hiệu + lan truyền (cách doocs)

In [8]:
def doocs_diff(n, delay, forget):
    m = (n << 1) + 10
    d = [0] * m; cnt = [0] * m; cnt[1] = 1
    for i in range(1, n + 1):
        if cnt[i]:
            d[i] += cnt[i]; d[i + forget] -= cnt[i]
            nxt = i + delay
            while nxt < i + forget:
                cnt[nxt] += cnt[i]; nxt += 1
    return sum(d[: n + 1]) % MOD

for n, dl, f, exp in examples:
    print(doocs_diff(n, dl, f), "OK" if doocs_diff(n, dl, f) == exp else "SAI")

5 OK
6 OK


Cách này "đẩy" (push): nhóm `cnt[i]` cộng vào mọi ngày chia sẻ của nó (O(forget − delay) mỗi nhóm), và đánh dấu vào mảng hiệu `d` khoảng thời gian `[i, i + forget)` mà nhóm còn nhớ. Tổng tiền tố của `d` tới ngày `n` = số người còn nhớ. O(n · (forget − delay)). Không lấy dư trong vòng lặp nên số rất lớn (Python vẫn đúng).

### 7.2 Cách 3 — Cửa sổ trượt O(n)

In [9]:
def sliding(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        share %= MOD
        new[d] = share
    return sum(new[max(1, n - forget + 1):]) % MOD

for n, d, f, exp in examples:
    print(sliding(n, d, f), "OK" if sliding(n, d, f) == exp else "SAI")

5 OK
6 OK


### 7.2b Bộ nhớ O(forget) bằng bộ đệm vòng

Chỉ cần nhớ `new` của `forget` ngày gần nhất (để trừ khi quên, và để cộng đáp án cuối cùng). Dùng `deque(maxlen = forget)`:

In [10]:
def ring_buffer(n, delay, forget):
    window = deque([0] * (forget - 1) + [1], maxlen=forget)   # new[d - forget + 1 .. d], ban đầu d = 1
    share = 0
    for d in range(2, n + 1):
        oldest = window[0]                                     # new[d - forget] (đã ra khỏi cửa sổ khi append)
        starter = window[-delay]                               # new[d - delay]
        share = (share + starter - oldest) % MOD
        window.append(share)                                   # tự động bỏ new[d - forget]
    return sum(window) % MOD

for n, d, f, exp in examples:
    print(ring_buffer(n, d, f), "OK" if ring_buffer(n, d, f) == exp else "SAI")

5 OK
6 OK


Tại đầu vòng lặp ngày `d`, `window` chứa `new[d − forget … d − 1]`; phần tử đầu là `new[d − forget]` (nhóm quên hôm nay), phần tử thứ `delay` tính từ cuối là `new[d − delay]` (nhóm bắt đầu chia sẻ hôm nay). Sau khi `append`, cửa sổ trượt sang `new[d − forget + 1 … d]` — đúng những người còn nhớ cuối ngày `d`.

### 7.3 Cách 4 — Tổng tiền tố

In [11]:
def prefix_sums(n, delay, forget):
    new = [0] * (n + 1); S = [0] * (n + 1)
    new[1] = 1; S[1] = 1
    for d in range(2, n + 1):
        hi, lo = d - delay, max(1, d - forget + 1)
        new[d] = (S[hi] - S[lo - 1]) % MOD if hi >= lo else 0
        S[d] = (S[d - 1] + new[d]) % MOD
    return (S[n] - S[max(0, n - forget)]) % MOD

for n, d, f, exp in examples:
    print(prefix_sums(n, d, f), "OK" if prefix_sums(n, d, f) == exp else "SAI")

5 OK
6 OK


### 7.4 Cách 5 — Hàng đợi `(ngày biết, số người)` — mô phỏng theo nhóm

In [12]:
def queue_groups(n, delay, forget):
    know = deque([(1, 1)])                    # nhóm còn nhớ: (ngày biết, số người)
    share = 0; waiting = deque([(1, 1)])      # nhóm chưa tới tuổi chia sẻ
    sharing = deque()                         # nhóm đang chia sẻ
    for d in range(2, n + 1):
        while waiting and waiting[0][0] + delay <= d:
            g = waiting.popleft(); sharing.append(g); share += g[1]
        while sharing and sharing[0][0] + forget <= d:
            share -= sharing.popleft()[1]
        while know and know[0][0] + forget <= d:
            know.popleft()
        share %= MOD
        if share:
            know.append((d, share)); waiting.append((d, share))
    return sum(c for _, c in know) % MOD

for n, d, f, exp in examples:
    print(queue_groups(n, d, f), "OK" if queue_groups(n, d, f) == exp else "SAI")

5 OK
6 OK


Ba hàng đợi tương ứng ba giai đoạn của một nhóm (chờ → chia sẻ → quên). Mỗi nhóm vào/ra mỗi hàng đợi một lần ⇒ O(n).

## 8. TRACE từng bước

### 8.1 Trace cửa sổ trượt trên ví dụ 2

In [13]:
n, delay, forget, _ = examples[1]
new = [0] * (n + 1); new[1] = 1; share = 0
for d in range(2, n + 1):
    add = new[d - delay] if d - delay >= 1 else 0
    rem = new[d - forget] if d - forget >= 1 else 0
    share += add - rem; new[d] = share
    print(f"ngày {d}: +{add} (tới tuổi chia sẻ), -{rem} (vừa quên) -> đang chia sẻ {share}, new[{d}] = {new[d]}")
print("còn nhớ cuối ngày", n, "= new[", max(1, n - forget + 1), "..", n, "] =", new[max(1, n - forget + 1):], "=", sum(new[max(1, n - forget + 1):]))

ngày 2: +1 (tới tuổi chia sẻ), -0 (vừa quên) -> đang chia sẻ 1, new[2] = 1
ngày 3: +1 (tới tuổi chia sẻ), -0 (vừa quên) -> đang chia sẻ 2, new[3] = 2
ngày 4: +2 (tới tuổi chia sẻ), -1 (vừa quên) -> đang chia sẻ 3, new[4] = 3
còn nhớ cuối ngày 4 = new[ 2 .. 4 ] = [1, 2, 3] = 6


Ngày 4: A (biết ngày 1) quên đúng ngày `1 + 3 = 4` ⇒ không chia sẻ; B (ngày 2), C, D (ngày 3) chia sẻ ⇒ 3 người mới. Còn nhớ: những người biết ngày 2, 3, 4: `1 + 2 + 3 = 6` ✔.

### 8.2 Trace mô phỏng từng người (đặt tên) cho ví dụ 1

In [14]:
n, delay, forget, _ = examples[0]
names = iter("ABCDEFGHIJKLMNOP"); people = [(next(names), 1)]
for d in range(2, n + 1):
    sharers = [p for p in people if p[1] + delay <= d < p[1] + forget]
    newbies = [(next(names), d) for _ in sharers]
    people += newbies
    alive = [p for p in people if p[1] + forget > d]
    print(f"ngày {d}: chia sẻ {[p[0] for p in sharers]} -> mới {[p[0] for p in newbies]}; còn nhớ {[p[0] for p in alive]} ({len(alive)})")

ngày 2: chia sẻ [] -> mới []; còn nhớ ['A'] (1)
ngày 3: chia sẻ ['A'] -> mới ['B']; còn nhớ ['A', 'B'] (2)
ngày 4: chia sẻ ['A'] -> mới ['C']; còn nhớ ['A', 'B', 'C'] (3)
ngày 5: chia sẻ ['B'] -> mới ['D']; còn nhớ ['B', 'C', 'D'] (3)
ngày 6: chia sẻ ['B', 'C'] -> mới ['E', 'F']; còn nhớ ['B', 'C', 'D', 'E', 'F'] (5)


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def peopleAwareOfSecret(self, n: int, delay: int, forget: int) -> int:
        MOD = 10**9 + 7
        new = [0] * (n + 1)              # new[d]: số người biết bí mật lần đầu vào ngày d
        new[1] = 1
        share = 0                        # số người đang chia sẻ trong ngày d
        for d in range(2, n + 1):
            if d - delay >= 1:
                share += new[d - delay]  # nhóm bắt đầu chia sẻ hôm nay
            if d - forget >= 1:
                share -= new[d - forget] # nhóm quên hôm nay: không chia sẻ nữa
            share %= MOD
            new[d] = share
        return sum(new[max(1, n - forget + 1):]) % MOD   # những người chưa quên

In [16]:
sol = Solution()
for n, d, f, exp in examples:
    got = sol.peopleAwareOfSecret(n, d, f)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
6 | kỳ vọng 6 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (nhóm).** Mọi người biết bí mật cùng ngày `j` có cùng lịch: chia sẻ đúng các ngày `[j + delay, j + forget − 1]`, nhớ đúng các ngày `[j, j + forget − 1]`. Vì vậy chỉ cần `new[j]`.

**Bổ đề 2 (truy hồi).** Mỗi người chia sẻ trong ngày `d` tạo đúng một người mới biết vào ngày `d`. Người biết ngày `j` chia sẻ ngày `d` ⇔ `j ∈ [d − forget + 1, d − delay]` ⇒ `new[d] = Σ_{j ∈ [d−forget+1, d−delay], j ≥ 1} new[j]` (người ban đầu là `new[1] = 1`; không ai khác biết ngày 1 vì `delay ≥ 1`).

**Bổ đề 3 (cửa sổ trượt).** Đặt `share(d)` = vế phải. Khi `d` tăng 1, cửa sổ `[d − forget + 1, d − delay]` thêm phần tử `d − delay` ở phải và bớt phần tử `d − forget` ở trái ⇒ `share(d) = share(d−1) + new[d − delay] − new[d − forget]` (bỏ các chỉ số `< 1`). Đúng như vòng lặp.

**Đáp án.** Người còn nhớ cuối ngày `n` ⇔ ngày biết `j` thỏa `j ≤ n ≤ j + forget − 1` ⇔ `j ∈ [n − forget + 1, n]`. ∎

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Từng người | O(số người · n) — hàm mũ | hàm mũ |
| 1b. DP cộng cửa sổ | O(n · forget) | O(n) |
| 2. Mảng hiệu + lan truyền (doocs) | O(n · (forget − delay)) | O(n) |
| 3. Cửa sổ trượt | O(n) | O(n) |
| 4. Tổng tiền tố | O(n) | O(n) |
| 5. Ba hàng đợi | O(n) | O(forget) |

In [17]:
res = []
for name, f in [("dp_quadratic", dp_quadratic), ("doocs_diff", doocs_diff), ("sliding", sliding),
                ("prefix_sums", prefix_sums), ("queue_groups", queue_groups), ("Solution", Solution().peopleAwareOfSecret)]:
    t0 = time.perf_counter(); res.append(f(1000, 1, 1000)); dt = time.perf_counter() - t0
    print(f"{name:<13} n=1000, delay=1, forget=1000: {dt*1000:7.1f} ms -> {res[-1]}")
assert len(set(res)) == 1

dp_quadratic  n=1000, delay=1, forget=1000:    21.0 ms -> 344211605


doocs_diff    n=1000, delay=1, forget=1000:    89.1 ms -> 344211605
sliding       n=1000, delay=1, forget=1000:     0.1 ms -> 344211605
prefix_sums   n=1000, delay=1, forget=1000:     0.2 ms -> 344211605
queue_groups  n=1000, delay=1, forget=1000:     0.5 ms -> 344211605
Solution      n=1000, delay=1, forget=1000:     0.1 ms -> 344211605


Trường hợp `delay = 1, forget = n` là worst cho các cách O(n · forget) và cũng làm số người tăng theo hàm mũ (gấp đôi mỗi ngày — `2^{999}` người trước khi lấy dư).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "dp_quadratic": dp_quadratic, "doocs_diff": doocs_diff, "sliding": sliding, "ring_buffer": ring_buffer,
       "prefix_sums": prefix_sums, "queue_groups": queue_groups, "Solution": Solution().peopleAwareOfSecret}
edge = examples + [
    (2, 1, 2, 2),          # ngày 2: A chia sẻ cho B; A quên ngày 3 -> cuối ngày 2 còn A, B
    (3, 1, 2, 2),          # mỗi người chia sẻ đúng 1 ngày rồi quên: chuỗi truyền tin
    (10, 1, 10, 512),      # gấp đôi mỗi ngày: 2^9
    (5, 4, 5, 2),          # A chia sẻ duy nhất ngày 5 (cho B), vẫn nhớ cuối ngày 5
]
for n, d, f, exp in edge:
    ref = brute(n, d, f)
    for name, fn in ALL.items():
        assert fn(n, d, f) == ref, (name, n, d, f)
    print(f"n={n:<3} delay={d} forget={f:<3} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

n=6   delay=2 forget=4   -> 5 | ghi trong bảng: 5 | khớp: True
n=4   delay=1 forget=3   -> 6 | ghi trong bảng: 6 | khớp: True
n=2   delay=1 forget=2   -> 2 | ghi trong bảng: 2 | khớp: True
n=3   delay=1 forget=2   -> 2 | ghi trong bảng: 2 | khớp: True
n=10  delay=1 forget=10  -> 512 | ghi trong bảng: 512 | khớp: True
n=5   delay=4 forget=5   -> 2 | ghi trong bảng: 2 | khớp: True


Hai case `(2, 1, 2)` và `(3, 1, 2)` đáng chú ý: với `forget = delay + 1`, mỗi người chia sẻ đúng **một** ngày (ngày `j + delay`) rồi quên vào ngày hôm sau. Ngày 2: A chia sẻ cho B **và** cuối ngày 2 A vẫn còn nhớ (A quên vào ngày 3) ⇒ **2** người — dễ tính nhầm thành 1 nếu quên rằng người chia sẻ vẫn còn nhớ trong ngày chia sẻ cuối cùng. Ngày 3: A quên, B chia sẻ cho C ⇒ B và C ⇒ 2.

### 12.2 Đối chiếu ngẫu nhiên với mô phỏng từng người (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(2, 16); f = rng.randint(2, n); d = rng.randint(1, f - 1)
    ref = brute(n, d, f)
    for name, fn in ALL.items():
        assert fn(n, d, f) == ref, (name, n, d, f)
print("400 bộ ngẫu nhiên OK")

400 bộ ngẫu nhiên OK


### 12.3 Tính chất: `delay = 1, forget = n` ⇒ `2^{n−1}`; `forget = delay + 1` ⇒ số người mới mỗi ngày ≤ 1 sau ngày `delay`

In [20]:
for n in range(2, 40):
    assert Solution().peopleAwareOfSecret(n, 1, n) == pow(2, n - 1, MOD)
for d in range(1, 8):
    for n in range(d + 1, 30):
        assert Solution().peopleAwareOfSecret(n, d, d + 1) in (1, 2)
print("tính chất OK")

tính chất OK


Với `forget = delay + 1`, tại mọi thời điểm chỉ có tối đa một người chia sẻ (mỗi người sinh đúng một người rồi quên) ⇒ "chuỗi truyền tin" một người — số người còn nhớ luôn là 1 hoặc 2.

### 12.4 Edge cực trị: `n = 1000` với nhiều cặp `(delay, forget)` — mọi cách O(n) khớp nhau

In [21]:
rng = random.Random(124)
for _ in range(60):
    f = rng.randint(2, 1000); d = rng.randint(1, f - 1)
    vals = {name: fn(1000, d, f) for name, fn in ALL.items() if name not in ("brute", "doocs_diff")}
    assert len(set(vals.values())) == 1, (d, f, vals)
print("60 cặp (delay, forget) với n = 1000: các cách khớp nhau; ví dụ (1, 1000) ->", Solution().peopleAwareOfSecret(1000, 1, 1000))

60 cặp (delay, forget) với n = 1000: các cách khớp nhau; ví dụ (1, 1000) -> 344211605


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về **tổng số người từng biết** (quên trừ người đã quên)

In [22]:
def wrong_total(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share
    return sum(new) % MOD                                  # SAI: tính cả người đã quên

for n, d, f, _ in examples:
    print("bản sai :", wrong_total(n, d, f), "| bản đúng:", Solution().peopleAwareOfSecret(n, d, f))

bản sai : 6 | bản đúng: 5
bản sai : 7 | bản đúng: 6


**Sửa:** chỉ cộng `new[d]` với `d ≥ n − forget + 1`.

### Lỗi 2 — Cho phép chia sẻ **vào ngày quên** (cửa sổ `[d − forget, d − delay]`)

In [23]:
def wrong_share_on_forget_day(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1
    for d in range(2, n + 1):
        new[d] = sum(new[j] for j in range(max(1, d - forget), d - delay + 1))   # SAI: thừa j = d - forget
    return sum(new[max(1, n - forget + 1):]) % MOD

for n, d, f, _ in examples:
    print("bản sai :", wrong_share_on_forget_day(n, d, f), "| bản đúng:", Solution().peopleAwareOfSecret(n, d, f))

bản sai : 6 | bản đúng: 5
bản sai : 7 | bản đúng: 6


**Vì sao sai:** đề: "không thể chia sẻ vào ngày quên". Ví dụ 1 ngày 5: A quên nên không chia sẻ — bản sai vẫn cho A chia sẻ.

### Lỗi 3 — Đếm người còn nhớ bằng cửa sổ lệch một (`[n − forget, n]`)

In [24]:
def wrong_alive_window(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share
    return sum(new[max(1, n - forget):]) % MOD             # SAI: người biết ngày n - forget đã quên đúng ngày n

for n, d, f, _ in examples:
    print("bản sai :", wrong_alive_window(n, d, f), "| bản đúng:", Solution().peopleAwareOfSecret(n, d, f))

bản sai : 5 | bản đúng: 5
bản sai : 7 | bản đúng: 6


**Vì sao sai:** người biết ngày `n − forget` quên đúng vào ngày `n` ⇒ cuối ngày `n` không còn biết. Ví dụ 1: thêm nhóm ngày 2 (`new[2] = 0`) nên **trùng hợp** đúng; ví dụ 2: thêm nhóm ngày 1 (A) ⇒ 7. Đây là lỗi "lệch một" điển hình giữa hai cách hiểu "quên sau `forget` ngày": nhớ trong các ngày `j … j + forget − 1` (đúng, theo ví dụ 1 — A biết ngày 1, quên ngày 5) hay `j … j + forget`. Luôn kiểm bằng ví dụ có lời giải từng ngày của đề.

### Lỗi 4 — Người mới bắt đầu chia sẻ ngay hôm sau (bỏ qua `delay`)

In [25]:
def wrong_no_delay(n, delay, forget):
    return Solution().peopleAwareOfSecret(n, 1, forget)    # SAI: luôn coi delay = 1

for n, d, f, _ in examples:
    print("bản sai :", wrong_no_delay(n, d, f), "| bản đúng:", Solution().peopleAwareOfSecret(n, d, f))

bản sai : 26 | bản đúng: 5
bản sai : 6 | bản đúng: 6


**Vì sao sai:** ví dụ 2 có `delay = 1` nên **trùng hợp** đúng; ví dụ 1 thì không.

### Lỗi 5 — Mô phỏng từng người cho `n = 1000`

In [26]:
def people_count_without_mod(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share
    return sum(new[max(1, n - forget + 1):])

for n in (20, 40, 1000):
    c = people_count_without_mod(n, 1, n)
    print(f"n={n}: số người thật (không lấy dư) có {len(str(c))} chữ số")

n=20: số người thật (không lấy dư) có 6 chữ số
n=40: số người thật (không lấy dư) có 12 chữ số
n=1000: số người thật (không lấy dư) có 301 chữ số


**Vì sao sai:** danh sách từng người cần lưu `2^{999}` phần tử — không thể. **Sửa:** đếm theo nhóm (`new[d]`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Các cá thể giống hệt nhau theo "ngày sinh" ⇒ đếm **theo nhóm** `new[d]` thay vì mô phỏng từng cá thể.
- Mỗi nhóm tác động lên một **đoạn** ngày ⇒ cửa sổ trượt / tổng tiền tố / mảng hiệu.
- Viết rõ ba mốc: bắt đầu chia sẻ `j + delay`, ngày cuối chia sẻ `j + forget − 1`, ngày quên `j + forget`.
- Đáp án "còn ... cuối ngày n" ⇒ một cửa sổ cuối cùng; "từng ... tới ngày n" ⇒ tổng tất cả.
- Khi `n` nhỏ (≤ 1000) mọi cách O(n·forget) đều qua; vẫn nên viết bản O(n) vì nó ngắn và là nền cho bản lũy thừa ma trận khi `n` lớn.
- Bài cùng họ: 1997 First Day Where You Have Been in All the Rooms, 2400, 70 Climbing Stairs (cửa sổ cố định 2), 1871 Jump Game VII (cửa sổ trượt trên DP), 1220 Count Vowels Permutation.

## 15. Biến thể của bài

### 15.1 Biến thể A — `n` rất lớn (`10¹⁸`): truy hồi tuyến tính bậc `forget` ⇒ lũy thừa ma trận

In [27]:
def matrix_version(n, delay, forget):
    K = forget                                      # trạng thái: new[d], new[d-1], ..., new[d-K+1]
    T = [[0] * K for _ in range(K)]
    for j in range(delay - 1, forget - 1):          # new[d+1] = sum new[d+1-k] với k = delay..forget-1
        T[0][j] = 1
    for i in range(1, K): T[i][i - 1] = 1
    def mul(A, B):
        return [[sum(A[i][k] * B[k][j] for k in range(K)) % MOD for j in range(K)] for i in range(K)]
    def mpow(A, e):
        R = [[int(i == j) for j in range(K)] for i in range(K)]
        while e:
            if e & 1: R = mul(R, A)
            A = mul(A, A); e >>= 1
        return R
    P = mpow(T, n - 1)                              # từ vector ngày 1 (new[1]=1, các ngày trước = 0)
    state = [P[i][0] for i in range(K)]             # new[n], new[n-1], ..., new[n-K+1]
    return sum(state) % MOD                         # đúng K = forget ngày cuối: những người còn nhớ

rng = random.Random(151)
for _ in range(200):
    n = rng.randint(2, 60); f = rng.randint(2, min(n, 8)); d = rng.randint(1, f - 1)
    assert matrix_version(n, d, f) == Solution().peopleAwareOfSecret(n, d, f), (n, d, f)
print(matrix_version(10**18, 2, 4), "(n = 10^18, delay = 2, forget = 4)")

21320 (n = 10^18, delay = 2, forget = 4)


`new[d + 1] = Σ_{k = delay}^{forget − 1} new[d + 1 − k]` — truy hồi tuyến tính bậc `forget − 1`; dùng vector `forget` phần tử để tiện cộng luôn cửa sổ "còn nhớ". Chi phí O(forget³ · log n).

### 15.2 Biến thể B — Số người **biết nhưng chưa được chia sẻ** cuối ngày `n`

In [28]:
def not_yet_sharing(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share % MOD
    return sum(new[max(1, n - delay + 1):]) % MOD   # biết trong delay ngày cuối

print(not_yet_sharing(6, 2, 4), "(ví dụ 1: D biết ngày 5, E và F biết ngày 6 -> chưa ai trong 3 người này chia sẻ được)")

3 (ví dụ 1: D biết ngày 5, E và F biết ngày 6 -> chưa ai trong 3 người này chia sẻ được)


### 15.3 Biến thể C — Ngày **đầu tiên** số người còn nhớ đạt ít nhất `X`

In [29]:
def first_day_at_least(X, delay, forget, limit=10**4):
    new = [0] * (limit + 1); new[1] = 1; share = 0; alive = 1
    if alive >= X: return 1
    for d in range(2, limit + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]; alive -= new[d - forget]
        new[d] = share; alive += share
        if alive >= X: return d
    return -1

print(first_day_at_least(5, 2, 4), first_day_at_least(6, 1, 3), first_day_at_least(10**6, 1, 3))

6 4 29


Ở đây không lấy dư (cần so sánh thật với `X`); số liệu tăng theo hàm mũ nên vòng lặp dừng sớm.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng người | định nghĩa | hàm mũ | hàm mũ | kiểm chứng n nhỏ |
| 1b. DP cộng cửa sổ | truy hồi trực tiếp | O(n·forget) | O(n) | dễ viết |
| 2. Mảng hiệu (doocs) | đẩy | O(n·(forget−delay)) | O(n) | trực quan |
| 3. Cửa sổ trượt | `share += … − …` | O(n) | O(n) | **mặc định** |
| 4. Tổng tiền tố | `S[hi] − S[lo−1]` | O(n) | O(n) | dễ kiểm tra biên |
| 5. Ba hàng đợi | giai đoạn nhóm | O(n) | O(forget) | mô phỏng theo sự kiện |
| 6. Ma trận | truy hồi tuyến tính | O(forget³ log n) | O(forget²) | n cực lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `peopleAwareOfSecret(3, 1, 2)` và `peopleAwareOfSecret(5, 2, 3)`?

**Bài 2 (tính tay):** Lập bảng `new[d]` và số người còn nhớ cho `n = 7, delay = 2, forget = 5`.

**Bài 3 (điền TODO):** Viết `bt3_todo(n, delay, forget)` trả về danh sách số người **còn nhớ** cuối mỗi ngày `1 … n`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(n, delay, forget)` cài cách **tổng tiền tố** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(n, delay, forget)` trả về **tổng số lần chia sẻ** đã xảy ra trong các ngày `1 … n` (mod 10⁹+7).

In [30]:
def bt3_todo(n, delay, forget):
    # TODO: số người còn nhớ cuối mỗi ngày
    pass

def bt4_todo(n, delay, forget):
    # TODO: tổng tiền tố
    pass

def bt5_todo(n, delay, forget):
    # TODO: tổng số lần chia sẻ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `forget = delay + 1` ⇒ chuỗi một người. Mức 2: ngày 3: A quên, B chia sẻ cho C ⇒ `2`. Mức 3: câu hai cũng có `forget = delay + 1`: `new[1..5] = 1, 0, 1, 0, 1` (mỗi người chỉ chia sẻ đúng ngày `j + 2`), còn nhớ là nhóm ngày 3…5: `1 + 0 + 1 = 2`.

**Bài 2.** Mức 1: cửa sổ chia sẻ của ngày `d` là `[d − 4, d − 2]`. Mức 2: `new[1..7] = 1, 0, 1, 1, 2, 2, 4` (ví dụ `new[7] = new[3] + new[4] + new[5] = 1 + 1 + 2`). Mức 3: cửa sổ còn nhớ `[3, 7]`: `1 + 1 + 2 + 2 + 4 = 10`.

**Bài 3.** Mức 1: tính `new` như bản nộp. Mức 2: với mỗi ngày `d`, cộng cửa sổ `[d − forget + 1, d]`. Mức 3: dùng tổng tiền tố cho O(n).

**Bài 4.** Mức 1: `S[d] = S[d−1] + new[d]`. Mức 2: `new[d] = S[d − delay] − S[d − forget]` (chặn chỉ số ≥ 0). Mức 3: đáp án `S[n] − S[n − forget]`.

**Bài 5.** Mức 1: mỗi lần chia sẻ tạo đúng một người mới. Mức 2: tổng số lần chia sẻ = tổng số người mới từ ngày 2. Mức 3: `Σ_{d=2}^{n} new[d]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `2`. **Bài 2:** `new = 1, 0, 1, 1, 2, 2, 4`, đáp án `10`.

(Lần đầu tính tay mình đã ghi `3` và `9` — assert bên dưới bắt được ngay; lỗi nằm ở việc quên rằng với `forget = delay + 1` mỗi người chỉ chia sẻ **một** ngày.)

In [31]:
assert Solution().peopleAwareOfSecret(3, 1, 2) == 2 and Solution().peopleAwareOfSecret(5, 2, 3) == 2
assert Solution().peopleAwareOfSecret(7, 2, 5) == 10

def bt3(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0; S = [0] * (n + 1); S[1] = 1
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share % MOD; S[d] = (S[d - 1] + new[d]) % MOD
    return [(S[d] - S[max(0, d - forget)]) % MOD for d in range(1, n + 1)]

def bt4(n, delay, forget):
    return prefix_sums(n, delay, forget)

def bt5(n, delay, forget):
    new = [0] * (n + 1); new[1] = 1; share = 0
    for d in range(2, n + 1):
        if d - delay >= 1: share += new[d - delay]
        if d - forget >= 1: share -= new[d - forget]
        new[d] = share % MOD
    return sum(new[2:]) % MOD

def bt5_brute(n, delay, forget):
    people = [1]; total = 0
    for d in range(2, n + 1):
        s = sum(1 for j in people if j + delay <= d < j + forget); total += s
        people += [d] * s
    return total

rng = random.Random(2000)
for _ in range(300):
    n = rng.randint(2, 14); f = rng.randint(2, n); d = rng.randint(1, f - 1)
    assert bt3(n, d, f) == [brute(k, d, f) if k >= 2 else 1 for k in range(1, n + 1)]
    assert bt4(n, d, f) == brute(n, d, f)
    assert bt5(n, d, f) == bt5_brute(n, d, f)
print(bt3(6, 2, 4), bt5(6, 2, 4))
print("Đáp án bài tập: tất cả assert OK")

[1, 1, 2, 3, 3, 5] 5
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
new = [0] * (n + 1); new[1] = 1; share = 0
for d in range(2, n + 1):
    if d - delay  >= 1: share += new[d - delay]    # bắt đầu chia sẻ
    if d - forget >= 1: share -= new[d - forget]   # quên: thôi chia sẻ
    share %= MOD; new[d] = share
return sum(new[max(1, n - forget + 1):]) % MOD     # còn nhớ cuối ngày n
```

**Checklist:**
- [ ] Đếm theo **nhóm ngày biết**, không mô phỏng từng người.
- [ ] Không chia sẻ vào **ngày quên** (cửa sổ `[d − forget + 1, d − delay]`).
- [ ] Đáp án chỉ gồm người **chưa quên**: ngày biết `≥ n − forget + 1`.
- [ ] Lấy dư mỗi bước (và `+ MOD` khi trừ ở ngôn ngữ khác).